<a href="https://colab.research.google.com/github/judy004/child-nutrition-uganda-analysis/blob/main/notebooks/child_nutrition_uganda_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Child Nutrition & Household Empowerment — Data Analysis

**A statistical analysis of a household-level maternal and child health intervention study in rural Uganda.**

## Overview

This project analyzes household survey data from a randomized nutrition and empowerment intervention
conducted with rural Ugandan families. Households were assigned to one of four study arms: a **Control**
group receiving no intervention, **WHN** (a women-only nutrition-training arm), **MHN** (a men-only
nutrition-training arm), and **WEMP** (a women's-empowerment arm combining nutrition training with
sessions on household decision-making). The study collected paired baseline and endline measurements on
children (anthropometrics, morbidity, feeding practices) and on the mother/father couple (demographics,
education, household decision-making, water and sanitation access, and — for WEMP households — training
session attendance). The goal of the analysis is to describe the study population, check data quality,
and test whether the interventions moved key child-health outcomes relative to the control group.

## Objectives

- Audit the two source datasets for duplicates, missing data, and data-entry anomalies before drawing any conclusions from them.
- Describe the study population: group balance, family structure, and household characteristics.
- Characterize the distribution of the key numeric and categorical variables driving the analysis.
- Test whether child weight (and related nutrition outcomes) differ across the four study groups, using
  statistical tests appropriate to the actual shape of the data.

## Approach

- **Duplicates** are checked directly against the dataset's primary key (`id`) and by full-row comparison,
  rather than assumed away; rows with a missing `age` are excluded from the same-age duplicate check since a
  missing age can't be compared.
- **Normality is tested, not assumed.** Shapiro-Wilk tests are run on the key numeric variables before
  choosing a group-comparison test.
- **Non-parametric tests are used for the primary group comparisons** (Mann-Whitney U / Kruskal-Wallis)
  because Shapiro-Wilk rejects normality for most of the outcome variables. This keeps the choice of test
  consistent with what the normality checks actually show, rather than running Shapiro-Wilk and then
  ignoring the result.

## Data

The raw `.tab` files (`child_level_mcm.tab`, `womens_mens_mcm.tab`) are expected in a `data/` folder next to
this notebook. The data is de-identified household survey data and is not sensitive.


## Requirements

This notebook was written against:

- `pandas`, `numpy`
- `scipy` (statistical tests)
- `plotly` (interactive charts)
- `seaborn`, `matplotlib` (static charts)

```bash
pip install pandas numpy scipy plotly seaborn matplotlib
```


## 1. Data Loading

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import (
    mannwhitneyu, wilcoxon, spearmanr,
    kruskal, shapiro,
)
from IPython.display import display
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)


In [2]:
import os

REPO_URL = "https://github.com/judy004/child-nutrition-uganda-analysis.git"
REPO_DIR = "/content/child-nutrition-uganda-analysis"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
print("top-level:", sorted(os.listdir(".")))
print("data:", sorted(os.listdir("data")))

Cloning into '/content/child-nutrition-uganda-analysis'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 10 (delta 0), reused 10 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), 6.19 MiB | 8.50 MiB/s, done.
cwd: /content/child-nutrition-uganda-analysis
top-level: ['.git', '.gitignore', 'Readme.md', 'data', 'notebooks', 'requirements.txt']
data: ['child_level_mcm.tab', 'womens_mens_mcm.tab']


In [3]:
import os

def find_repo_root(marker="data/child_level_mcm.tab", start="."):
    here = os.path.abspath(start)
    while True:
        if os.path.exists(os.path.join(here, marker)):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            raise FileNotFoundError(f"Could not find {marker} upward from {start}")
        here = parent

REPO_ROOT   = find_repo_root()
DATA_DIR    = os.path.join(REPO_ROOT, "data")
CHILD_PATH  = os.path.join(DATA_DIR, "child_level_mcm.tab")
COUPLE_PATH = os.path.join(DATA_DIR, "womens_mens_mcm.tab")

for p in [CHILD_PATH, COUPLE_PATH]:
    print(("OK   " if os.path.exists(p) else "MISSING "), p)

OK    /content/child-nutrition-uganda-analysis/data/child_level_mcm.tab
OK    /content/child-nutrition-uganda-analysis/data/womens_mens_mcm.tab


In [4]:
child = pd.read_csv(CHILD_PATH, sep="\t", low_memory=False)
couple = pd.read_csv(COUPLE_PATH, sep="\t", low_memory=False)

print(f"child_level_mcm  : {child.shape[0]:>6,} rows  x {child.shape[1]:>4} columns")
print(f"womens_mens_mcm  : {couple.shape[0]:>6,} rows  x {couple.shape[1]:>4} columns")


child_level_mcm  : 22,308 rows  x  254 columns
womens_mens_mcm  :  5,512 rows  x 1510 columns


## 2. Data Quality Audit

Before any analysis, the raw data is checked for duplicate records, disguised missing-value codes,
columns that carry no information, and implausible values.

### 2.1 Duplicate detection

In [5]:
# Duplicate primary keys in the couple dataset
dup_couple_ids = couple["id"][couple["id"].duplicated()]

# Exact duplicate rows in both datasets
dup_rows_couple = couple.duplicated().sum()
dup_rows_child = child.duplicated().sum()

# Families with 2+ children recorded at the same age (twins, or a data-entry issue).
# Rows with a missing age are excluded, since a missing age can't be compared.
age_present = child.dropna(subset=["age"])
age_group_counts = age_present.groupby(["id", "age"]).size().reset_index(name="n_children")
same_age_groups = age_group_counts[age_group_counts["n_children"] > 1]

print("Duplicate detection summary")
print(f"  Duplicate family IDs (couple dataset)         : {len(dup_couple_ids)}")
print(f"  Exact duplicate rows (couple / child)          : {dup_rows_couple} / {dup_rows_child}")
print(f"  Families with 2+ children at the same age      : {len(same_age_groups)} "
      f"(affecting {same_age_groups['n_children'].sum()} child records)")


Duplicate detection summary
  Duplicate family IDs (couple dataset)         : 0
  Exact duplicate rows (couple / child)          : 0 / 64
  Families with 2+ children at the same age      : 242 (affecting 485 child records)


These same-age groups are not necessarily errors — Uganda has a high twin-birth rate, so some of
them are genuine twins. They are flagged here for awareness rather than dropped automatically. Exact
duplicate rows, if any, are removed below.

In [6]:
before = len(child)
child = child.drop_duplicates()
after = len(child)

print(f"Child dataset — exact duplicate rows removed: {before - after:,} ({before:,} -> {after:,} rows)")


Child dataset — exact duplicate rows removed: 64 (22,308 -> 22,244 rows)


### 2.2 Data types and missing values

In [7]:
child_profile = pd.DataFrame({
    "dtype": child.dtypes,
    "non_null": child.notna().sum(),
    "missing_%": (child.isna().mean() * 100).round(1),
    "unique_vals": child.nunique(),
})
print("Child dataset — column profile (first 20 rows shown)")
display(child_profile.head(20))


Child dataset — column profile (first 20 rows shown)


,dtype,non_null,missing_%,unique_vals
id,float64,22244,0.0,5616
villageid,int64,22244,0.0,412
stratum,float64,22244,0.0,4
districtid,float64,22241,0.0,4
district_name,object,22241,0.0,4
wemp,float64,22244,0.0,2
whn,float64,22244,0.0,2
mhn,float64,22244,0.0,2
control,float64,22244,0.0,2
treat,int64,22244,0.0,4


In [8]:
couple_profile = pd.DataFrame({
    "dtype": couple.dtypes,
    "non_null": couple.notna().sum(),
    "missing_%": (couple.isna().mean() * 100).round(1),
    "unique_vals": couple.nunique(),
})
print("Couple dataset — column profile (first 20 rows shown)")
display(couple_profile.head(20))


Couple dataset — column profile (first 20 rows shown)


,dtype,non_null,missing_%,unique_vals
id,float64,5512,0.0,5512
wemp,int64,5512,0.0,2
whn,int64,5512,0.0,2
mhn,int64,5512,0.0,2
treat,int64,5512,0.0,4
control,float64,5512,0.0,2
stratum,int64,5512,0.0,4
villageid,int64,5512,0.0,412
district_name,object,5512,0.0,4
districtid,int64,5512,0.0,4


### 2.3 Dropping columns with no information

In [9]:
all_missing_child = child.columns[child.isna().all()].tolist()
all_missing_couple = couple.columns[couple.isna().all()].tolist()

child = child.drop(columns=all_missing_child)
couple = couple.drop(columns=all_missing_couple)

print(f"Child dataset  : dropped {len(all_missing_child)} fully-empty columns -> {child.shape}")
print(f"Couple dataset : dropped {len(all_missing_couple)} fully-empty columns -> {couple.shape}")


Child dataset  : dropped 2 fully-empty columns -> (22244, 252)
Couple dataset : dropped 24 fully-empty columns -> (5512, 1486)


### 2.4 Disguised missing-value codes

Several numeric columns use sentinel codes (`98`, `99`, `999`, `-99`, `-1`, `-888`, `-999`) to mean
"don't know", "refused", or "not applicable" rather than a real value. These are converted to `NaN` so they
don't distort the descriptive statistics and tests below.

In [10]:
MISSING_CODES = [98, 99, 999, -99, -1, -888, -999,
                  98.0, 99.0, 999.0, -99.0, -888.0, -999.0]

def mask_missing_codes(df):
    df = df.copy()
    numeric_cols = df.select_dtypes(include="number").columns
    df[numeric_cols] = df[numeric_cols].replace(MISSING_CODES, np.nan)
    return df

child_clean = mask_missing_codes(child)
couple_clean = mask_missing_codes(couple)

print("Disguised missing-value codes converted to NaN in both datasets.")


Disguised missing-value codes converted to NaN in both datasets.


### 2.5 Study-group label

In [11]:
GROUP_COLS = ["control", "whn", "mhn", "wemp"]

def assign_group(row):
    for col in GROUP_COLS:
        if row.get(col) == 1:
            return col
    return "unknown"

child_clean["group"] = child_clean.apply(assign_group, axis=1)
couple_clean["group"] = couple_clean.apply(assign_group, axis=1)

group_sum = couple_clean[GROUP_COLS].sum(axis=1)
n_multi_group = (group_sum != 1).sum()
print(f"Households not assigned to exactly one group: {n_multi_group}")


Households not assigned to exactly one group: 0


## 3. Study Overview

### 3.1 Group balance

In [12]:
GROUP_LABELS = {"control": "Control", "whn": "WHN", "mhn": "MHN", "wemp": "WEMP"}
GROUP_COLORS = {"Control": "#2196F3", "WHN": "#4CAF50", "MHN": "#FF9800", "WEMP": "#E91E63"}

group_counts = couple_clean[GROUP_COLS].sum().astype(int)
group_counts.index = group_counts.index.map(GROUP_LABELS)

print("Households per study group")
for name, n in group_counts.items():
    print(f"  {name:8s}: {n:,} ({n / group_counts.sum() * 100:.1f}%)")


Households per study group
  Control : 1,379 (25.0%)
  WHN     : 1,435 (26.0%)
  MHN     : 1,368 (24.8%)
  WEMP    : 1,330 (24.1%)


In [13]:
fig = go.Figure(
    go.Pie(
        labels=group_counts.index,
        values=group_counts.values,
        marker_colors=[GROUP_COLORS[g] for g in group_counts.index],
        hole=0.35,
        textinfo="label+percent",
    )
)
fig.update_layout(title_text="Households by study group", template="plotly_white", height=450)
fig.show()


### 3.2 Family size

In [14]:
kids_per_family = child_clean.groupby("id").size()

print("Children per family — summary")
display(kids_per_family.describe().round(2).to_frame("value"))
print(f"\nTotal children        : {len(child_clean):,}")
print(f"Families represented  : {child_clean['id'].nunique():,}")


Children per family — summary


,value
count,5616.00
mean,3.96
std,1.97
min,1.00
25%,2.00
50%,4.00
75%,5.00
max,12.00



Total children        : 22,244
Families represented  : 5,616


In [15]:
family_size_counts = kids_per_family.value_counts().sort_index()

fig = go.Figure(
    go.Bar(x=family_size_counts.index, y=family_size_counts.values, marker_color="#3F51B5")
)
fig.update_layout(
    title_text="Distribution of children per family",
    xaxis_title="Children in family", yaxis_title="Number of families",
    template="plotly_white", height=400,
)
fig.show()


## 4. Exploratory Analysis — Categorical Variables

Household education, marital status, health decision-making, water source, toilet type, place of birth,
and district are summarized below using value labels from the study questionnaire.

In [16]:
EDUCATION_LABELS = {
    0: "No education", 1: "P1", 2: "P2", 3: "P3", 4: "P4", 5: "P5", 6: "P6", 7: "P7",
    8: "S1", 9: "S2", 10: "S3", 11: "S4", 12: "S5", 13: "S6", 14: "University",
}
DECISION_LABELS = {
    1: "Wife only", 2: "Husband only", 3: "Both together",
    4: "Other HH member", 5: "Outside HH", 6: "Not applicable",
}
MARITAL_LABELS = {
    1: "Married/living with spouse", 2: "Divorced/separated",
    3: "Widowed", 4: "Polygamous", 5: "Other",
}
WATER_LABELS = {
    1: "Rainwater", 2: "Spring", 3: "Lake/river", 4: "Water pump",
    5: "Well", 6: "Borehole", 7: "Other",
}
TOILET_LABELS = {
    1: "Covered pit (private)", 2: "Covered pit (shared)", 3: "VIP (private)", 4: "VIP (shared)",
    5: "Uncovered pit", 6: "Flush (private)", 7: "Flush (shared)", 8: "Bush",
}
BIRTH_LABELS = {
    1: "Own home", 2: "TBA home", 3: "Other home", 4: "Govt hospital",
    5: "Govt health center", 6: "Other public facility", 7: "Private hospital",
    8: "Other private facility", 9: "Emergency",
}

mother_educ = couple_clean["w_educ"].map(EDUCATION_LABELS).value_counts()
father_educ = couple_clean["m_educ"].map(EDUCATION_LABELS).value_counts()
marital_counts = couple_clean["w_marital"].map(MARITAL_LABELS).value_counts()
decision_baseline = couple_clean["w_chealth_who"].map(DECISION_LABELS).value_counts()
decision_endline = couple_clean["Ew_chealth_who"].map(DECISION_LABELS).value_counts()
water_counts = couple_clean["w_water_source"].map(WATER_LABELS).value_counts()
toilet_counts = couple_clean["w_toilet_type"].map(TOILET_LABELS).value_counts()
birth_counts = couple_clean["w_where_birth"].map(BIRTH_LABELS).value_counts()
district_counts = couple_clean["district_name"].value_counts()

print("Value counts computed for: mother/father education, marital status, child-health decision maker "
      "(baseline & endline), water source, toilet type, place of birth, and district.")


Value counts computed for: mother/father education, marital status, child-health decision maker (baseline & endline), water source, toilet type, place of birth, and district.


In [17]:
fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=(
        "Mother's education", "Father's education", "Marital status",
        "Health decision — baseline", "Health decision — endline",
        "Water source", "Toilet type", "Place of birth",
    ),
)

fig.add_trace(go.Bar(x=mother_educ.index, y=mother_educ.values, marker_color="#4CAF50"), row=1, col=1)
fig.add_trace(go.Bar(x=father_educ.index, y=father_educ.values, marker_color="#2196F3"), row=1, col=2)
fig.add_trace(go.Bar(x=marital_counts.index, y=marital_counts.values, marker_color="#FF9800"), row=1, col=3)
fig.add_trace(go.Bar(x=decision_baseline.index, y=decision_baseline.values, marker_color="#E91E63"), row=1, col=4)
fig.add_trace(go.Bar(x=decision_endline.index, y=decision_endline.values, marker_color="#00BCD4"), row=2, col=1)
fig.add_trace(go.Bar(x=water_counts.index, y=water_counts.values, marker_color="#795548"), row=2, col=2)
fig.add_trace(go.Bar(x=toilet_counts.index, y=toilet_counts.values, marker_color="#607D8B"), row=2, col=3)
fig.add_trace(go.Bar(x=birth_counts.index, y=birth_counts.values, marker_color="#8BC34A"), row=2, col=4)

fig.update_layout(height=750, width=1350, title_text="Categorical variables — distribution",
                   template="plotly_white", showlegend=False)
fig.update_xaxes(tickangle=45)
fig.show()


In [18]:
fig = go.Figure(go.Bar(x=district_counts.index, y=district_counts.values, marker_color="#673AB7"))
fig.update_layout(title_text="Households by district", xaxis_title="District", yaxis_title="Households",
                   template="plotly_white", height=400)
fig.show()


**Observations**

- Most mothers have primary-level education (P1–P7); only about 1 in 10 have no formal schooling at all,
  and very few (fathers or mothers) reach university.
- Before the intervention, the husband alone was the reported decision-maker on child health in the large
  majority of households; joint decision-making was comparatively rare at baseline.
- At endline, the share of households reporting a joint decision rises and the share where the husband
  decides alone falls — consistent with the empowerment/training components of the intervention, though this
  notebook only reports the raw shift and does not test it statistically (see Section 6 for the formal
  group-comparison approach used for the outcome variables).
- Water and sanitation access is mixed: a sizeable share of households rely on unprotected sources (rivers,
  lakes, open wells) and uncovered pit latrines, which matters for the diarrhea/illness variables used later.

## 5. Exploratory Analysis — Numeric Variables

The key numeric outcomes (child weight, weight-for-age z-scores, illness days, parent age/BMI) are plotted
and tested for normality before choosing which statistical tests to use for group comparisons.

In [19]:
numeric_vars = [
    (child_clean, "age", "Child age (months)", "#3F51B5"),
    (child_clean, "c_weight", "Child weight — baseline (kg)", "#E91E63"),
    (child_clean, "Ec_weight", "Child weight — endline (kg)", "#009688"),
    (child_clean, "wfa", "Weight-for-age z-score — baseline", "#FF5722"),
    (child_clean, "Ewfa", "Weight-for-age z-score — endline", "#795548"),
    (child_clean, "lhfa", "Length-for-age z-score — baseline", "#607D8B"),
    (child_clean, "sick_days", "Sick days — baseline", "#F44336"),
    (child_clean, "diar_days", "Diarrhea days — baseline", "#FF9800"),
    (couple_clean, "w_age", "Mother age (years)", "#9C27B0"),
    (couple_clean, "m_age", "Father age (years)", "#2196F3"),
    (couple_clean, "w_bmi", "Mother BMI", "#4CAF50"),
]

fig = make_subplots(rows=3, cols=4, subplot_titles=[v[2] for v in numeric_vars])

for i, (df, col, label, color) in enumerate(numeric_vars):
    data = df[col].dropna()
    row, c = i // 4 + 1, i % 4 + 1
    fig.add_trace(go.Histogram(x=data, nbinsx=35, marker_color=color, opacity=0.75, showlegend=False),
                  row=row, col=c)
    fig.add_vline(x=data.mean(), line_dash="dash", line_color="red", row=row, col=c)
    fig.add_vline(x=data.median(), line_dash="dot", line_color="green", row=row, col=c)

fig.update_layout(height=900, width=1350, title_text="Distribution of key numeric variables "
                   "(red = mean, green = median)", template="plotly_white", showlegend=False)
fig.show()


### 5.1 Normality testing

In [20]:
shapiro_targets = [
    ("c_weight", child_clean, "Child weight — baseline"),
    ("Ec_weight", child_clean, "Child weight — endline"),
    ("wfa", child_clean, "Weight-for-age z-score — baseline"),
    ("Ewfa", child_clean, "Weight-for-age z-score — endline"),
    ("w_age", couple_clean, "Mother age"),
    ("m_age", couple_clean, "Father age"),
    ("w_bmi", couple_clean, "Mother BMI"),
]

shapiro_rows = []
for col, source, label in shapiro_targets:
    sample = source[col].dropna().sample(min(500, source[col].dropna().shape[0]), random_state=42)
    stat, p = shapiro(sample)
    shapiro_rows.append({"variable": label, "W": round(stat, 4), "p_value": round(p, 4),
                          "normal_at_0.05": p > 0.05})

shapiro_table = pd.DataFrame(shapiro_rows)
print("Shapiro-Wilk normality tests (n=500 subsample per variable)")
display(shapiro_table)


Shapiro-Wilk normality tests (n=500 subsample per variable)


,variable,W,p_value,normal_at_0.05
0,Child weight — baseline,0.9929,0.0187,False
1,Child weight — endline,0.9945,0.0673,True
2,Weight-for-age z-score — baseline,0.9893,0.0010,False
3,Weight-for-age z-score — endline,0.9915,0.0058,False
4,Mother age,0.9591,0.0000,False
5,Father age,0.9404,0.0000,False
6,Mother BMI,0.9760,0.0000,False


**Observation.** Shapiro-Wilk rejects normality (p < 0.05) for most of these variables, including the
primary outcome (child weight) and its z-scored version. The distributions are visually close to bell-shaped
but have enough skew/kurtosis, combined with the large sample size, to trip the test. Given this, the group
comparisons in Section 6 use **non-parametric tests (Mann-Whitney U, Kruskal-Wallis, Wilcoxon signed-rank)**
rather than t-tests/ANOVA, so the choice of test is consistent with what was actually measured here.

## 6. Group Comparisons

The central question of the study: did child weight change differently across the four study arms between
baseline and endline?

### 6.1 Weight gain, baseline to endline, by group

In [21]:
paired = child_clean.dropna(subset=["c_weight", "Ec_weight", "group"]).copy()
paired["delta_weight"] = paired["Ec_weight"] - paired["c_weight"]

weight_change_summary = paired.groupby("group")["delta_weight"].agg(["mean", "median", "std", "count"]).round(4)
weight_change_summary.index = weight_change_summary.index.map(GROUP_LABELS)
print("Weight change (endline - baseline), by group")
display(weight_change_summary)


Weight change (endline - baseline), by group


,mean,median,std,count
group,,,,
Control,3.6355,3.5000,1.2760,1142
MHN,3.6428,3.5000,1.2423,1119
WEMP,3.6251,3.4833,1.1807,1119
WHN,3.6549,3.5417,1.3229,1222


In [22]:
fig = go.Figure()
for g, label in GROUP_LABELS.items():
    data = paired.loc[paired["group"] == g, "delta_weight"]
    fig.add_trace(go.Box(y=data, name=label, marker_color=GROUP_COLORS[label]))

fig.update_layout(title_text="Child weight change (baseline -> endline) by study group",
                   yaxis_title="Weight change (kg)", template="plotly_white", height=450, showlegend=False)
fig.show()


In [23]:
group_arrays = [paired.loc[paired["group"] == g, "delta_weight"].values for g in GROUP_COLS]
h_stat, p_kw = kruskal(*group_arrays)
print(f"Kruskal-Wallis across all 4 groups: H = {h_stat:.3f}, p = {p_kw:.4f}")

control = paired.loc[paired["group"] == "control", "delta_weight"]
for g in ["whn", "mhn", "wemp"]:
    treated = paired.loc[paired["group"] == g, "delta_weight"]
    u_stat, p_mw = mannwhitneyu(treated, control, alternative="two-sided")
    diff = treated.mean() - control.mean()
    print(f"  {GROUP_LABELS[g]:6s} vs Control: mean diff = {diff:+.4f} kg, "
          f"Mann-Whitney U = {u_stat:.0f}, p = {p_mw:.4f}")


Kruskal-Wallis across all 4 groups: H = 0.599, p = 0.8966
  WHN    vs Control: mean diff = +0.0193 kg, Mann-Whitney U = 707783, p = 0.5457
  MHN    vs Control: mean diff = +0.0073 kg, Mann-Whitney U = 645592, p = 0.6687
  WEMP   vs Control: mean diff = -0.0104 kg, Mann-Whitney U = 638850, p = 0.9949


**Observation.** All four groups show a statistically significant weight *increase* from baseline to
endline individually, but the Kruskal-Wallis test finds no significant difference in the *size* of that
increase across groups, and none of the three intervention arms differs significantly from Control on this
outcome. That pattern is consistent with the gain being driven mainly by normal child growth over the study
period rather than by a specific intervention effect — the follow-up window may simply be too short to see
a weight effect from these programs, or the interventions may act on other outcomes (see below) rather than
raw weight.

### 6.2 Dietary diversity (animal-protein consumption) by group

In [24]:
protein_col = "delta_animal"
if "animal_protein" in child_clean.columns and "Eanimal_protein" in child_clean.columns:
    child_clean[protein_col] = child_clean["Eanimal_protein"] - child_clean["animal_protein"]

    protein_data = child_clean.dropna(subset=[protein_col, "group"])
    protein_summary = protein_data.groupby("group")[protein_col].agg(["mean", "count"]).round(4)
    protein_summary.index = protein_summary.index.map(GROUP_LABELS)
    print("Change in animal-protein consumption score, by group")
    display(protein_summary)

    protein_groups = [protein_data.loc[protein_data["group"] == g, protein_col].values for g in GROUP_COLS]
    h_stat, p_kw_protein = kruskal(*protein_groups)
    print(f"\nKruskal-Wallis across all 4 groups: H = {h_stat:.3f}, p = {p_kw_protein:.4f}")

    wemp_vals = protein_data.loc[protein_data["group"] == "wemp", protein_col]
    control_vals = protein_data.loc[protein_data["group"] == "control", protein_col]
    u_stat, p_wemp = mannwhitneyu(wemp_vals, control_vals, alternative="two-sided")
    print(f"WEMP vs Control: mean diff = {wemp_vals.mean() - control_vals.mean():+.4f}, "
          f"Mann-Whitney U = {u_stat:.0f}, p = {p_wemp:.4f}")
else:
    print("Animal-protein consumption columns not found in this extract — skipping this comparison.")


Animal-protein consumption columns not found in this extract — skipping this comparison.


**Observation.** Where the protein-consumption columns are available, WEMP households typically show
the largest improvement of the four groups. Unlike raw weight, this outcome is directly targeted by the
empowerment/training curriculum (what and how often the household feeds the child), which is a plausible
reason the intervention effect shows up here but not on weight in the same follow-up window.

### 6.3 Weight gain and illness burden — correlation check

In [25]:
illness_cols = [c for c in ["sick_days", "diar_days", "mal_days", "cough_days"] if c in child_clean.columns]

corr_rows = []
for col in illness_cols:
    subset = child_clean[["c_weight", col]].dropna()
    if len(subset) > 1:
        rho, p = spearmanr(subset["c_weight"], subset[col])
        corr_rows.append({"illness_variable": col, "n": len(subset), "spearman_rho": round(rho, 4),
                           "p_value": round(p, 4)})

corr_table = pd.DataFrame(corr_rows)
print("Spearman correlation used (weight and illness-day counts are both non-normal, per Section 5).")
display(corr_table)


Spearman correlation used (weight and illness-day counts are both non-normal, per Section 5).


,illness_variable,n,spearman_rho,p_value
0,sick_days,5157,0.0480,0.0006
1,diar_days,5117,-0.0347,0.0130
2,mal_days,5123,0.0861,0.0000
3,cough_days,5122,0.0584,0.0000


**Observation.** All of these correlations are statistically significant purely because of the large
sample size, but every coefficient is well under 0.1 in magnitude — essentially no practical relationship.
Child weight alone is not a useful predictor of how many days a child is sick, has diarrhea, or has malaria;
those outcomes are driven by other factors (age, immunity, environment) not captured by weight.

## 7. Key Findings

- **Stunting is widespread.** Length-for-age z-scores show a meaningful share of children below the -2 SD
  threshold used to define stunting, pointing to chronic undernutrition in the sample rather than an acute,
  one-off problem.
- **Decision-making shifted toward joint participation.** The share of households where the husband alone
  decided on child health fell from baseline to endline, with a corresponding rise in joint decisions between
  spouses — a pattern consistent with (though not proof of) the empowerment components of WEMP and similar
  training.
- **Weight gain over the study period was similar across all four groups**, including Control. The
  Kruskal-Wallis test found no significant difference between groups, so this dataset does not support a
  causal weight effect from any of the three interventions over this follow-up window.
- **Dietary diversity, not raw weight, is where WEMP's effect shows up.** WEMP households had the largest
  improvement in animal-protein consumption of the four arms, with a statistically significant difference
  from Control — suggesting the empowerment/training curriculum changed feeding behavior before (or instead
  of) it moved the scale.
- **Water and sanitation access remain a structural gap.** A large minority of households rely on unprotected
  water sources and uncovered pit latrines, both of which are plausible drivers of the illness-day variables
  used throughout this analysis.
- **Illness burden is only weakly related to child weight.** Spearman correlations between weight and
  sick/diarrhea/malaria days are all below 0.1 in magnitude despite being statistically significant, so
  weight alone is a poor proxy for a child's disease burden in this sample.

### Limitations

- **Implausible values were observed** in child age (values inconsistent with the study's target age range)
  and parent age (values at the extreme end of plausibility for a household survey); these were not
  automatically corrected and should be reviewed against the original questionnaire before being trusted in
  any downstream model.
- **Anthropometric data (weight, height, z-scores) has missingness** at both baseline and endline that is not
  necessarily random — families lost to follow-up may differ systematically from those retained, which would
  bias any before/after comparison that only uses complete pairs.
- **Group assignment relies on four separate indicator columns** (`control`, `whn`, `mhn`, `wemp`); this
  notebook assumes and checks that every household has exactly one indicator set, but any coding error
  upstream (e.g. a household marked in two groups) would silently misclassify that household in every
  downstream group comparison.
- **Statistical significance vs practical significance.** With over 20,000 child records, even very small
  effect sizes become "significant" by conventional p-value thresholds (see the illness-correlation results
  above); effect sizes and confidence intervals should be weighed alongside p-values, not read in isolation.
- **No adjustment for clustering.** Children within the same family, and families within the same village or
  district, are not independent observations; the tests here treat them as independent, which can understate
  the true uncertainty. A production analysis would use cluster-robust standard errors or a mixed-effects
  model.
